# Innings Pitched Model

### Checking for Correlations with new_IP

In [1]:
import pandas as pd
# Creating initial Data Frame
df = pd.read_csv("Clean_2015_Pitching_Data.csv", sep = ",")

# Eliminating String Features to be able to run Correlation Analysis
# Eliminating all target features except new_IP
new_IP_df = df.drop(['NAME', 'ID', 'new_yahoo', 'new_strikeout', 'new_win', 'new_loss', 'new_hit', 'new_earned_run', 'new_walk'], axis=1)

In [3]:
# Calculate the correlation matrix
correlation_matrix = new_IP_df.corr()

# Select the target variable column
target_variable = "new_IP" 

# Get correlations with the target variable
target_correlations = correlation_matrix[target_variable].sort_values(ascending=False)

# Print the correlations (optional)
print(target_correlations)

new_IP               1.000000
strikeout            0.314918
last_year_Yahoo      0.305171
p_swinging_strike    0.268233
IP                   0.267397
                       ...   
on_base_plus_slg    -0.227658
woba                -0.233316
xwoba               -0.238806
on_base_percent     -0.246783
xobp                -0.253199
Name: new_IP, Length: 97, dtype: float64


### Model 1 - Single Feature Model

In [7]:
from sklearn.model_selection import train_test_split
train_set, test_set = train_test_split(new_IP_df,
test_size=0.2, random_state=123)
print(len(train_set), len(test_set))

from sklearn.linear_model import LinearRegression
reg = LinearRegression()

X = train_set[['on_base_percent']]
y = train_set['new_IP']
reg.fit(X, y)

print("The bias is " , reg.intercept_)
print("The feature coefficients are ", reg.coef_)
print("The score for the training set is", reg.score(X,y))

# Check the performance on the test set
X_test = test_set[['on_base_percent']]
y_test = test_set['new_IP']
print("The score for the test set is", reg.score(X_test,y_test))

332 83
The bias is  244.94817383712117
The feature coefficients are  [-248.37439223]
The score for the training set is 0.07584178513202966
The score for the test set is -0.009769604530009302


#### Single Feature Model Performance
| Feature   | Training   | Test    |
| -----     | -----      | -----   |
| xobp | 0.07 | 0.02 |
| strikeout | 0.12 | 0.01 |
| last_year_Yahoo | 0.11 | 0.01 |
| IP | 0.09 | -.03 |




### Model 2 - Multi-Feature Model

In [11]:
X = train_set[['xobp','strikeout', 'p_swinging_strike']]
y = train_set['new_IP']
reg.fit(X, y)

print("The bias is " , reg.intercept_)
print("The feature coefficients are ", reg.coef_)
print("The score for the training set is", reg.score(X,y))

# Check the performance on the test set
X_test = test_set[['xobp','strikeout', 'p_swinging_strike']]
y_test = test_set['new_IP']
print("The score for the test set is", reg.score(X_test,y_test))

The bias is  160.51216319691068
The feature coefficients are  [-67.90295717   0.29857118  -0.06815864]
The score for the training set is 0.12711222302557956
The score for the test set is -0.005471235543777819


#### Multi-Feature Model Performance
| # Features | Features | Training   | Test    |
| -----     | -----      | -----   | ----- |
| 2 | strikeout, last_year_Yahoo | 0.12 | 0.01 |
| 2 | strikeout, xobp | 0.12 | 0.01 |
| 2 | xobp, IP | 0.12 | 0.00 |
| 3 | xobp, strikeout, p_swinging_strike | 0.13 | -0.01 |


### Model 3 - Decision Tree Model

In [13]:
from sklearn.tree import DecisionTreeRegressor
from sklearn.model_selection import train_test_split

# Separate features (X) and target variable (y)
X = new_IP_df.drop(['new_IP'], axis =1)
y = new_IP_df['new_IP']

# Split data into training and testing sets (test_size=0.2 for 20% test data)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

In [18]:
from sklearn.metrics import confusion_matrix
from sklearn.metrics import accuracy_score, f1_score
from sklearn.metrics import precision_score, recall_score

# Create the decision tree regression model
model = DecisionTreeRegressor(max_depth=4)

# Train the model on the training data
model.fit(X_train, y_train)

# Make predictions on testing data
y_pred = model.predict(X_test)

from sklearn.metrics import r2_score
from sklearn.metrics import mean_squared_error

mse = mean_squared_error(y_test, y_pred)
print("Mean Squared Error:", mse)
r2 = r2_score(y_test, y_pred)
print("R-squared:", r2)

Mean Squared Error: 974.0417080937704
R-squared: -0.4408541425249455


#### Decision Tree Results
| Depth | MSE   | R2    |
| ----- | ----- | ----- |
| 3     | 748 | -.10 |
| 2     | 706 | -.04 |
| 1     | 615 | .09 |

### Model 4 - LASSO Model

In [28]:
from sklearn.model_selection import train_test_split
from sklearn.linear_model import Lasso

# Create the LASSO model with alpha (regularization parameter)
model = Lasso(alpha=200)  # Adjust alpha as needed

# Train the model on the training data
model.fit(X_train, y_train)

# Make predictions on testing data
y_pred = model.predict(X_test)

# Evaluate model performance (e.g., using mean squared error)
mse = mean_squared_error(y_test, y_pred)
print("Mean Squared Error:", mse)
r2 = r2_score(y_test, y_pred)
print("R-squared:", r2)

Mean Squared Error: 640.2537564662973
R-squared: 0.05290269440616968


#### Lasso Alpha Analysis
| Alpha | MSE   | R2    |
| ----- | ----- | ----- |
| .1    | 853 | -.26 |
| 1     | 781 | -.16 |
| 10    | 689 | -.02 |
| 100   | 641 | .05 |

# Final Results

| Method | Specifics   | R2   |
| ----- | ----- | ----- |
